# Budapest Spa Demo - Data Ingestion\n\nThis notebook generates the POC database for Aqua Serenity Spas competitive intelligence demo.

In [1]:
import os
from snowflake.snowpark import Session

session = Session.builder.config("connection_name", os.getenv("SNOWFLAKE_CONNECTION_NAME", "oregon_tp")).create()
print(f"Connected as: {session.get_current_user()}")
print(f"Role: {session.get_current_role()}")
print(f"Warehouse: {session.get_current_warehouse()}")

Connected as: "admin"
Role: "ACCOUNTADMIN"
Warehouse: "AI_WH"


## Step 1: Create Database and Schema

In [2]:
session.sql("CREATE OR REPLACE DATABASE BUDAPEST_SPA_DEMO").collect()
session.sql("CREATE OR REPLACE SCHEMA BUDAPEST_SPA_DEMO.ANALYTICS").collect()
session.sql("USE SCHEMA BUDAPEST_SPA_DEMO.ANALYTICS").collect()
print("Database and schema created.")

Database and schema created.


## Step 2: Create Tables

In [ ]:
session.sql("""
CREATE OR REPLACE TABLE DIM_SPAS (
    SPA_ID INT PRIMARY KEY,
    SPA_NAME VARCHAR(200) NOT NULL,
    LATITUDE FLOAT,
    LONGITUDE FLOAT,
    DISTRICT VARCHAR(10),
    NEIGHBORHOOD VARCHAR(100),
    SPA_TYPE VARCHAR(50),
    PRICE_TIER VARCHAR(20),
    IS_OWNED BOOLEAN DEFAULT FALSE,
    OPENING_YEAR INT,
    CAPACITY INT
)
""").collect()

session.sql("""
CREATE OR REPLACE TABLE DIM_TREATMENTS (
    TREATMENT_ID INT PRIMARY KEY,
    TREATMENT_NAME VARCHAR(100) NOT NULL,
    CATEGORY VARCHAR(50),
    AVG_DURATION_MIN INT,
    AVG_PRICE_HUF INT
)
""").collect()

session.sql("""
CREATE OR REPLACE TABLE DIM_FACILITIES (
    FACILITY_ID INT PRIMARY KEY,
    FACILITY_NAME VARCHAR(100) NOT NULL,
    CATEGORY VARCHAR(50)
)
""").collect()

session.sql("""
CREATE OR REPLACE TABLE BRIDGE_SPA_TREATMENTS (
    SPA_ID INT,
    TREATMENT_ID INT,
    PRIMARY KEY (SPA_ID, TREATMENT_ID)
)
""").collect()

session.sql("""
CREATE OR REPLACE TABLE BRIDGE_SPA_FACILITIES (
    SPA_ID INT,
    FACILITY_ID INT,
    PRIMARY KEY (SPA_ID, FACILITY_ID)
)
""").collect()

session.sql("""
CREATE OR REPLACE TABLE FACT_SPA_METRICS (
    SPA_ID INT,
    YEAR_MONTH DATE,
    TOTAL_VISITS INT,
    REVENUE_HUF INT,
    AVG_SPEND_PER_VISIT INT,
    OCCUPANCY_RATE FLOAT,
    NEW_CUSTOMERS INT,
    RETURNING_CUSTOMERS INT,
    PRIMARY KEY (SPA_ID, YEAR_MONTH)
)
""").collect()

session.sql("""
CREATE OR REPLACE TABLE REVIEWS (
    REVIEW_ID INT PRIMARY KEY,
    SPA_ID INT NOT NULL,
    REVIEWER_NAME VARCHAR(100),
    REVIEWER_LOCATION VARCHAR(100),
    RATING INT,
    REVIEW_DATE DATE,
    REVIEW_TITLE VARCHAR(500),
    REVIEW_TEXT VARCHAR(4000),
    VISIT_TYPE VARCHAR(50)
)
""").collect()

print("All tables created successfully.")

All tables created successfully.


## Step 3: Generate Spa Location Data

In [4]:
from snowflake.snowpark.types import StructType, StructField, StringType, FloatType, IntegerType, BooleanType
import pandas as pd

spas_data = [
    # Our 3 Aqua Serenity Locations (is_owned = True)
    (1, "Aqua Serenity Castle District", 47.4979, 19.0402, "I", "Varnegyed", "Wellness Spa", "Premium", True, 2019, 80),
    (2, "Aqua Serenity City Park", 47.5151, 19.0833, "XIV", "Varosliget", "Wellness Spa", "Mid-Range", True, 2018, 150),
    (3, "Aqua Serenity Buda Hills", 47.5017, 18.9683, "XII", "Svabhegy", "Boutique Spa", "Luxury", True, 2021, 40),
    
    # Historic Thermal Baths (8)
    (4, "Szechenyi Thermal Bath", 47.5188, 19.0822, "XIV", "Varosliget", "Thermal Bath", "Mid-Range", False, 1913, 400),
    (5, "Gellert Thermal Bath", 47.4835, 19.0522, "XI", "Gellert Hill", "Thermal Bath", "Premium", False, 1918, 300),
    (6, "Rudas Thermal Bath", 47.4866, 19.0487, "I", "Taban", "Thermal Bath", "Mid-Range", False, 1550, 200),
    (7, "Kiraly Thermal Bath", 47.5072, 19.0370, "II", "Vizivaros", "Thermal Bath", "Budget", False, 1565, 150),
    (8, "Lukacs Thermal Bath", 47.5179, 19.0351, "II", "Vizivaros", "Thermal Bath", "Budget", False, 1894, 180),
    (9, "Palatinus Strand", 47.5289, 19.0458, "XIII", "Margaret Island", "Thermal Bath", "Budget", False, 1919, 500),
    (10, "Veli Bej Bath", 47.5139, 19.0359, "II", "Vizivaros", "Thermal Bath", "Premium", False, 1574, 80),
    (11, "Dandar Thermal Bath", 47.4756, 19.0875, "IX", "Ferencvaros", "Thermal Bath", "Budget", False, 1930, 120),
    
    # Hotel Spas (10)
    (12, "Four Seasons Gresham Spa", 47.4996, 19.0461, "V", "Belvaros", "Hotel Spa", "Luxury", False, 2004, 60),
    (13, "Corinthia Royal Spa", 47.5034, 19.0633, "VII", "Erzsebetvaros", "Hotel Spa", "Luxury", False, 1896, 100),
    (14, "Aria Hotel Harmony Spa", 47.5015, 19.0513, "V", "Belvaros", "Hotel Spa", "Luxury", False, 2015, 50),
    (15, "Kempinski Spa Budapest", 47.4972, 19.0565, "V", "Belvaros", "Hotel Spa", "Luxury", False, 2010, 70),
    (16, "Hilton Budapest Spa", 47.5018, 19.0358, "I", "Varnegyed", "Hotel Spa", "Premium", False, 1976, 80),
    (17, "InterContinental Spa", 47.4964, 19.0480, "V", "Belvaros", "Hotel Spa", "Premium", False, 2000, 60),
    (18, "Marriott Budapest Spa", 47.4941, 19.0509, "V", "Belvaros", "Hotel Spa", "Premium", False, 2005, 55),
    (19, "New York Palace Spa", 47.4994, 19.0695, "VII", "Erzsebetvaros", "Hotel Spa", "Luxury", False, 2006, 45),
    (20, "The Ritz-Carlton Spa", 47.4958, 19.0556, "V", "Belvaros", "Hotel Spa", "Luxury", False, 2016, 50),
    (21, "Sofitel Budapest Spa", 47.4912, 19.0486, "V", "Belvaros", "Hotel Spa", "Premium", False, 2008, 65),
    
    # Wellness/Day Spas (8)
    (22, "Mandala Day Spa", 47.5041, 19.0625, "VI", "Terezvaros", "Day Spa", "Premium", False, 2012, 40),
    (23, "Balance Wellness Center", 47.4889, 19.0721, "VIII", "Jozsefvaros", "Wellness Spa", "Mid-Range", False, 2015, 80),
    (24, "Oasis Urban Spa", 47.5105, 19.0557, "VI", "Terezvaros", "Day Spa", "Mid-Range", False, 2017, 50),
    (25, "Zenith Wellness Budapest", 47.4778, 19.0456, "XI", "Ujbuda", "Wellness Spa", "Mid-Range", False, 2016, 90),
    (26, "Serenity Day Spa", 47.5198, 19.0601, "XIII", "Angyalfold", "Day Spa", "Budget", False, 2019, 35),
    (27, "Urban Retreat Budapest", 47.4923, 19.0832, "IX", "Ferencvaros", "Day Spa", "Mid-Range", False, 2018, 45),
    (28, "Bliss Wellness Buda", 47.4856, 19.0167, "XI", "Sasad", "Wellness Spa", "Premium", False, 2014, 70),
    (29, "Harmony Spa Pest", 47.5067, 19.0899, "XIV", "Zuglo", "Wellness Spa", "Budget", False, 2020, 60),
    
    # Boutique Specialty Spas (6)
    (30, "Thai Orchid Spa", 47.5012, 19.0712, "VII", "Erzsebetvaros", "Boutique Spa", "Premium", False, 2013, 25),
    (31, "Ayurveda Budapest", 47.4945, 19.0389, "I", "Krisztinavaros", "Boutique Spa", "Luxury", False, 2016, 20),
    (32, "Hammam Turkish Bath", 47.5089, 19.0482, "V", "Lipotvaros", "Boutique Spa", "Premium", False, 2018, 30),
    (33, "Nordic Sauna World", 47.5234, 19.0756, "XIV", "Herminamezö", "Boutique Spa", "Mid-Range", False, 2019, 40),
    (34, "Japanese Onsen Budapest", 47.4801, 19.0612, "IX", "Ferencvaros", "Boutique Spa", "Luxury", False, 2021, 25),
    (35, "Salt Cave Wellness", 47.5156, 19.0423, "II", "Rozsadomb", "Boutique Spa", "Budget", False, 2017, 35),
]

pdf = pd.DataFrame(spas_data, columns=[
    "SPA_ID", "SPA_NAME", "LATITUDE", "LONGITUDE", "DISTRICT", "NEIGHBORHOOD", 
    "SPA_TYPE", "PRICE_TIER", "IS_OWNED", "OPENING_YEAR", "CAPACITY"
])

df = session.create_dataframe(pdf)
df.write.mode("overwrite").save_as_table("DIM_SPAS")
print(f"Inserted {session.table('DIM_SPAS').count()} spas")

Inserted 35 spas


## Step 4: Generate Treatments and Facilities

In [5]:
treatments_data = [
    (1, "Thermal Bath", "Thermal/Bath", 60, 8000),
    (2, "Cold Plunge", "Thermal/Bath", 15, 3000),
    (3, "Mineral Bath", "Thermal/Bath", 45, 10000),
    (4, "Mud Treatment", "Thermal/Bath", 30, 12000),
    (5, "Swedish Massage", "Massage", 60, 18000),
    (6, "Thai Massage", "Massage", 90, 25000),
    (7, "Hot Stone Massage", "Massage", 75, 22000),
    (8, "Deep Tissue Massage", "Massage", 60, 20000),
    (9, "Aromatherapy Massage", "Massage", 60, 19000),
    (10, "Classic Facial", "Facial", 45, 15000),
    (11, "Anti-Aging Facial", "Facial", 60, 28000),
    (12, "Hydrating Facial", "Facial", 50, 18000),
    (13, "Body Scrub", "Body", 30, 12000),
    (14, "Body Wrap", "Body", 45, 16000),
    (15, "Detox Treatment", "Body", 60, 22000),
    (16, "Couples Treatment", "Specialty", 90, 45000),
    (17, "Medical Massage", "Specialty", 45, 25000),
    (18, "Hydrotherapy", "Specialty", 30, 15000),
    (19, "Salt Therapy", "Specialty", 45, 10000),
    (20, "Float Therapy", "Specialty", 60, 18000),
]

pdf_treatments = pd.DataFrame(treatments_data, columns=["TREATMENT_ID", "TREATMENT_NAME", "CATEGORY", "AVG_DURATION_MIN", "AVG_PRICE_HUF"])
df_treatments = session.create_dataframe(pdf_treatments)
df_treatments.write.mode("overwrite").save_as_table("DIM_TREATMENTS")
print(f"Inserted {session.table('DIM_TREATMENTS').count()} treatments")

facilities_data = [
    (1, "Indoor Pool", "Pools"),
    (2, "Outdoor Pool", "Pools"),
    (3, "Thermal Pool", "Pools"),
    (4, "Wave Pool", "Pools"),
    (5, "Lap Pool", "Pools"),
    (6, "Finnish Sauna", "Heat"),
    (7, "Steam Room", "Heat"),
    (8, "Infrared Sauna", "Heat"),
    (9, "Salt Room", "Heat"),
    (10, "Relaxation Room", "Relaxation"),
    (11, "Rooftop Terrace", "Relaxation"),
    (12, "Garden", "Relaxation"),
    (13, "Private Cabins", "Relaxation"),
    (14, "Restaurant", "Amenities"),
    (15, "Cafe", "Amenities"),
    (16, "Fitness Center", "Amenities"),
    (17, "Locker Room", "Amenities"),
    (18, "Kids Area", "Amenities"),
]

pdf_facilities = pd.DataFrame(facilities_data, columns=["FACILITY_ID", "FACILITY_NAME", "CATEGORY"])
df_facilities = session.create_dataframe(pdf_facilities)
df_facilities.write.mode("overwrite").save_as_table("DIM_FACILITIES")
print(f"Inserted {session.table('DIM_FACILITIES').count()} facilities")

Inserted 20 treatments
Inserted 18 facilities


## Step 5: Create Spa-Treatment and Spa-Facility Mappings

In [6]:
import random
random.seed(42)

spas_df = session.table("DIM_SPAS").to_pandas()
treatments_df = session.table("DIM_TREATMENTS").to_pandas()
facilities_df = session.table("DIM_FACILITIES").to_pandas()

treatment_rules = {
    "Thermal Bath": (12, 18),
    "Hotel Spa": (10, 14),
    "Wellness Spa": (8, 12),
    "Day Spa": (5, 9),
    "Boutique Spa": (4, 7),
}

facility_rules = {
    "Thermal Bath": (14, 18),
    "Hotel Spa": (10, 14),
    "Wellness Spa": (8, 12),
    "Day Spa": (5, 8),
    "Boutique Spa": (4, 6),
}

spa_treatments = []
spa_facilities = []

for _, spa in spas_df.iterrows():
    spa_id = spa["SPA_ID"]
    spa_type = spa["SPA_TYPE"]
    is_owned = spa["IS_OWNED"]
    
    t_min, t_max = treatment_rules.get(spa_type, (6, 10))
    f_min, f_max = facility_rules.get(spa_type, (6, 10))
    
    if is_owned:
        t_min = max(3, t_min - 3)
        t_max = max(5, t_max - 3)
        f_min = max(3, f_min - 3)
        f_max = max(5, f_max - 3)
    
    num_treatments = random.randint(t_min, t_max)
    num_facilities = random.randint(f_min, f_max)
    
    selected_treatments = random.sample(list(treatments_df["TREATMENT_ID"]), min(num_treatments, len(treatments_df)))
    selected_facilities = random.sample(list(facilities_df["FACILITY_ID"]), min(num_facilities, len(facilities_df)))
    
    for tid in selected_treatments:
        spa_treatments.append((int(spa_id), int(tid)))
    for fid in selected_facilities:
        spa_facilities.append((int(spa_id), int(fid)))

pdf_spa_treatments = pd.DataFrame(spa_treatments, columns=["SPA_ID", "TREATMENT_ID"])
df_spa_treatments = session.create_dataframe(pdf_spa_treatments)
df_spa_treatments.write.mode("overwrite").save_as_table("BRIDGE_SPA_TREATMENTS")

pdf_spa_facilities = pd.DataFrame(spa_facilities, columns=["SPA_ID", "FACILITY_ID"])
df_spa_facilities = session.create_dataframe(pdf_spa_facilities)
df_spa_facilities.write.mode("overwrite").save_as_table("BRIDGE_SPA_FACILITIES")

print(f"Created {len(spa_treatments)} spa-treatment mappings")
print(f"Created {len(spa_facilities)} spa-facility mappings")

Created 361 spa-treatment mappings
Created 361 spa-facility mappings


## Step 6: Generate Monthly Spa Metrics

In [7]:
from datetime import datetime
from dateutil.relativedelta import relativedelta
import random

random.seed(42)

spas_df = session.table("DIM_SPAS").to_pandas()

price_tier_multipliers = {"Budget": 0.7, "Mid-Range": 1.0, "Premium": 1.4, "Luxury": 2.0}
spa_type_base_visits = {"Thermal Bath": 8000, "Hotel Spa": 1500, "Wellness Spa": 2000, "Day Spa": 1200, "Boutique Spa": 600}
avg_spend = {"Budget": 8000, "Mid-Range": 15000, "Premium": 25000, "Luxury": 45000}

start_month = datetime(2023, 1, 1)
end_month = datetime(2026, 2, 1)

metrics_data = []
current_month = start_month

while current_month <= end_month:
    for _, spa in spas_df.iterrows():
        spa_id = int(spa["SPA_ID"])
        spa_type = spa["SPA_TYPE"]
        price_tier = spa["PRICE_TIER"]
        capacity = int(spa["CAPACITY"])
        is_owned = spa["IS_OWNED"]
        
        base_visits = spa_type_base_visits.get(spa_type, 1500)
        
        month_num = current_month.month
        if month_num in [6, 7, 8]:
            seasonality = 1.3
        elif month_num in [12, 1]:
            seasonality = 1.15
        elif month_num in [3, 4]:
            seasonality = 0.85
        else:
            seasonality = 1.0
        
        noise = random.uniform(0.85, 1.15)
        
        if is_owned:
            performance_factor = random.uniform(0.75, 0.95)
        else:
            performance_factor = random.uniform(0.9, 1.1)
        
        monthly_visits = int(base_visits * seasonality * noise * performance_factor / 12)
        monthly_visits = min(monthly_visits, capacity * 30)
        
        base_spend = avg_spend.get(price_tier, 15000)
        avg_spend_per_visit = int(base_spend * random.uniform(0.9, 1.1))
        revenue = monthly_visits * avg_spend_per_visit
        
        max_monthly_capacity = capacity * 30
        occupancy_rate = round(min(monthly_visits / max_monthly_capacity, 0.95), 2)
        
        new_customers = int(monthly_visits * random.uniform(0.3, 0.5))
        returning_customers = monthly_visits - new_customers
        
        metrics_data.append((
            spa_id,
            current_month.strftime("%Y-%m-01"),
            monthly_visits,
            revenue,
            avg_spend_per_visit,
            occupancy_rate,
            new_customers,
            returning_customers
        ))
    
    current_month += relativedelta(months=1)

pdf_metrics = pd.DataFrame(metrics_data, columns=[
    "SPA_ID", "YEAR_MONTH", "TOTAL_VISITS", "REVENUE_HUF", 
    "AVG_SPEND_PER_VISIT", "OCCUPANCY_RATE", "NEW_CUSTOMERS", "RETURNING_CUSTOMERS"
])

df_metrics = session.create_dataframe(pdf_metrics)
df_metrics.write.mode("overwrite").save_as_table("FACT_SPA_METRICS")
print(f"Generated {session.table('FACT_SPA_METRICS').count()} monthly metrics records")

Generated 1330 monthly metrics records


## Step 7: Generate 250 TripAdvisor-Style Reviews

In [8]:
from datetime import datetime, timedelta
import random

random.seed(42)

spas_df = session.table("DIM_SPAS").to_pandas()

reviewer_names = [
    "TravelJunkie", "SpaLover", "WellnessSeeker", "BudapestVisitor", "RelaxationPro",
    "JohnM", "SarahK", "MarkT", "EmmaW", "DavidL", "SophieB", "ChrisP", "AnnaH",
    "MichaelR", "LauraS", "ThomasG", "JuliaF", "PeterN", "KateM", "RobertC",
    "SpaExplorer", "ThermalFan", "EuroTraveler", "HealthyLife", "ZenMaster",
]

reviewer_locations = [
    "London, UK", "Berlin, Germany", "New York, USA", "Paris, France", "Rome, Italy",
    "Amsterdam, Netherlands", "Vienna, Austria", "Madrid, Spain", "Warsaw, Poland",
    "Prague, Czech Republic", "Stockholm, Sweden", "Dublin, Ireland", "Munich, Germany",
]

visit_types = ["Solo", "Couple", "Family", "Friends", "Business"]
visit_weights = [15, 40, 20, 20, 5]

start_date = datetime(2023, 1, 1)
end_date = datetime(2026, 2, 28)
date_range = (end_date - start_date).days

popular_spa_ids = spas_df[spas_df["SPA_TYPE"] == "Thermal Bath"]["SPA_ID"].tolist()
owned_spa_ids = spas_df[spas_df["IS_OWNED"] == True]["SPA_ID"].tolist()

base_reviews_per_spa = {spa_id: 6 for spa_id in spas_df["SPA_ID"]}
for spa_id in popular_spa_ids[:4]:
    base_reviews_per_spa[spa_id] = 12
for spa_id in popular_spa_ids[4:]:
    base_reviews_per_spa[spa_id] = 10
for spa_id in owned_spa_ids:
    base_reviews_per_spa[spa_id] = 8

total_reviews = sum(base_reviews_per_spa.values())
scale_factor = 250 / total_reviews
for spa_id, count in base_reviews_per_spa.items():
    base_reviews_per_spa[spa_id] = max(3, int(count * scale_factor))

def get_rating_for_spa(spa_type, is_owned):
    if is_owned:
        weights = [5, 12, 25, 33, 25]
    elif spa_type == "Thermal Bath":
        weights = [2, 5, 15, 35, 43]
    elif spa_type == "Hotel Spa":
        weights = [3, 7, 18, 32, 40]
    else:
        weights = [5, 10, 20, 30, 35]
    return random.choices([1, 2, 3, 4, 5], weights=weights)[0]

reviews_metadata = []
for _, spa in spas_df.iterrows():
    spa_id = int(spa["SPA_ID"])
    spa_name = spa["SPA_NAME"]
    spa_type = spa["SPA_TYPE"]
    price_tier = spa["PRICE_TIER"]
    district = spa["DISTRICT"]
    is_owned = spa["IS_OWNED"]
    
    num_reviews = base_reviews_per_spa.get(spa_id, 5)
    
    for _ in range(num_reviews):
        rating = get_rating_for_spa(spa_type, is_owned)
        reviewer_name = random.choice(reviewer_names) + str(random.randint(1, 999))
        reviewer_location = random.choice(reviewer_locations)
        review_date = (start_date + timedelta(days=random.randint(0, date_range))).strftime("%Y-%m-%d")
        visit_type = random.choices(visit_types, weights=visit_weights)[0]
        
        reviews_metadata.append((
            spa_id, spa_name, spa_type, price_tier, district, is_owned,
            reviewer_name, reviewer_location, rating, review_date, visit_type
        ))

random.shuffle(reviews_metadata)
reviews_metadata = reviews_metadata[:250]

pdf_meta = pd.DataFrame(reviews_metadata, columns=[
    "SPA_ID", "SPA_NAME", "SPA_TYPE", "PRICE_TIER", "DISTRICT", "IS_OWNED",
    "REVIEWER_NAME", "REVIEWER_LOCATION", "RATING", "REVIEW_DATE", "VISIT_TYPE"
])
session.create_dataframe(pdf_meta).write.mode("overwrite").save_as_table("REVIEWS_STAGING")
print(f"Created staging table with {len(reviews_metadata)} review metadata records")

Created staging table with 221 review metadata records


In [9]:
session.sql("""
CREATE OR REPLACE TABLE REVIEWS AS
SELECT 
    ROW_NUMBER() OVER (ORDER BY REVIEW_DATE) AS REVIEW_ID,
    SPA_ID,
    REVIEWER_NAME,
    REVIEWER_LOCATION,
    RATING,
    REVIEW_DATE::DATE AS REVIEW_DATE,
    SNOWFLAKE.CORTEX.COMPLETE(
        'claude-sonnet-4-5',
        'Generate a short TripAdvisor review title (5-10 words) for a ' || RATING || '-star review of a ' 
        || SPA_TYPE || ' called "' || SPA_NAME || '" in Budapest District ' || DISTRICT || '. '
        || 'Price tier: ' || PRICE_TIER || '. Visit type: ' || VISIT_TYPE || '. '
        || CASE 
            WHEN RATING >= 4 THEN 'The reviewer loved it.'
            WHEN RATING = 3 THEN 'The reviewer had mixed feelings.'
            ELSE 'The reviewer was disappointed.'
           END
        || ' Return ONLY the title, no quotes or explanation.'
    ) AS REVIEW_TITLE,
    SNOWFLAKE.CORTEX.COMPLETE(
        'claude-sonnet-4-5',
        'Write a realistic TripAdvisor review (80-120 words) for a ' || RATING || '-star visit to '
        || SPA_NAME || ', a ' || SPA_TYPE || ' in Budapest District ' || DISTRICT || '. '
        || 'Price tier: ' || PRICE_TIER || '. Visitor from: ' || REVIEWER_LOCATION || '. '
        || 'Visit type: ' || VISIT_TYPE || '. '
        || CASE 
            WHEN IS_OWNED THEN 'This is a newer wellness spa brand. Common issues: wait times, limited treatments, outdated facilities. Positives: friendly staff, clean, good location.'
            WHEN SPA_TYPE = 'Thermal Bath' THEN 'This is a famous historic Hungarian thermal bath with authentic thermal pools and Ottoman/Art Nouveau architecture.'
            WHEN SPA_TYPE = 'Hotel Spa' THEN 'This is a luxury hotel spa with premium amenities.'
            ELSE 'This is a modern spa/wellness center.'
           END
        || CASE 
            WHEN RATING = 5 THEN ' Write an enthusiastic glowing review praising specific features.'
            WHEN RATING = 4 THEN ' Write a positive review with one minor complaint.'
            WHEN RATING = 3 THEN ' Write a mediocre review mentioning both positives and negatives.'
            WHEN RATING = 2 THEN ' Write a negative review with specific complaints.'
            ELSE ' Write a very negative review detailing multiple problems.'
           END
        || ' Be specific and authentic. Do not use generic phrases. Return ONLY the review text.'
    ) AS REVIEW_TEXT,
    VISIT_TYPE
FROM REVIEWS_STAGING
""").collect()

print(f"Generated {session.table('REVIEWS').count()} AI-written reviews")

Generated 221 AI-written reviews


In [10]:
session.sql("DROP TABLE IF EXISTS REVIEWS_STAGING").collect()
print("Cleaned up staging table.")

Cleaned up staging table.


## Step 8: Validate the Data

In [11]:
print("=== Data Validation Summary ===\n")

print("Row Counts:")
for table in ["DIM_SPAS", "DIM_TREATMENTS", "DIM_FACILITIES", "BRIDGE_SPA_TREATMENTS", "BRIDGE_SPA_FACILITIES", "FACT_SPA_METRICS", "REVIEWS"]:
    count = session.table(table).count()
    print(f"  {table}: {count}")

print("\n--- Rating Distribution ---")
session.sql("""
    SELECT 
        RATING,
        COUNT(*) AS COUNT,
        ROUND(COUNT(*) * 100.0 / SUM(COUNT(*)) OVER(), 1) AS PERCENTAGE
    FROM REVIEWS
    GROUP BY RATING
    ORDER BY RATING DESC
""").show()

print("\n--- Average Ratings: Our Spas vs Competitors ---")
session.sql("""
    SELECT 
        CASE WHEN s.IS_OWNED THEN 'Our Spas' ELSE 'Competitors' END AS OWNERSHIP,
        ROUND(AVG(r.RATING), 2) AS AVG_RATING,
        COUNT(*) AS REVIEW_COUNT
    FROM REVIEWS r
    JOIN DIM_SPAS s ON r.SPA_ID = s.SPA_ID
    GROUP BY OWNERSHIP
    ORDER BY OWNERSHIP
""").show()

print("\n--- Top 5 Spas by Rating (min 5 reviews) ---")
session.sql("""
    SELECT 
        s.SPA_NAME,
        s.SPA_TYPE,
        s.IS_OWNED,
        ROUND(AVG(r.RATING), 2) AS AVG_RATING,
        COUNT(*) AS REVIEW_COUNT
    FROM REVIEWS r
    JOIN DIM_SPAS s ON r.SPA_ID = s.SPA_ID
    GROUP BY s.SPA_ID, s.SPA_NAME, s.SPA_TYPE, s.IS_OWNED
    HAVING COUNT(*) >= 5
    ORDER BY AVG_RATING DESC
    LIMIT 5
""").show()

print("\n--- Monthly Metrics Sample (Our Spas, 2025) ---")
session.sql("""
    SELECT 
        s.SPA_NAME,
        m.YEAR_MONTH,
        m.TOTAL_VISITS,
        m.REVENUE_HUF,
        m.OCCUPANCY_RATE
    FROM FACT_SPA_METRICS m
    JOIN DIM_SPAS s ON m.SPA_ID = s.SPA_ID
    WHERE s.IS_OWNED = TRUE AND m.YEAR_MONTH >= '2025-01-01'
    ORDER BY s.SPA_NAME, m.YEAR_MONTH
    LIMIT 6
""").show()

print("\n--- Sample Reviews ---")
session.sql("""
    SELECT 
        s.SPA_NAME,
        r.RATING,
        r.REVIEW_TITLE,
        LEFT(r.REVIEW_TEXT, 100) || '...' AS REVIEW_PREVIEW
    FROM REVIEWS r
    JOIN DIM_SPAS s ON r.SPA_ID = s.SPA_ID
    ORDER BY RANDOM()
    LIMIT 3
""").show()

print("\n=== Validation Complete ===")

=== Data Validation Summary ===

Row Counts:
  DIM_SPAS: 35
  DIM_TREATMENTS: 20
  DIM_FACILITIES: 18
  BRIDGE_SPA_TREATMENTS: 361
  BRIDGE_SPA_FACILITIES: 361
  FACT_SPA_METRICS: 1330
  REVIEWS: 221

--- Rating Distribution ---
-------------------------------------
|"RATING"  |"COUNT"  |"PERCENTAGE"  |
-------------------------------------
|5         |83       |37.6          |
|4         |84       |38.0          |
|3         |36       |16.3          |
|2         |9        |4.1           |
|1         |9        |4.1           |
-------------------------------------


--- Average Ratings: Our Spas vs Competitors ---
-----------------------------------------------
|"OWNERSHIP"  |"AVG_RATING"  |"REVIEW_COUNT"  |
-----------------------------------------------
|Competitors  |4.07          |200             |
|Our Spas     |3.48          |21              |
-----------------------------------------------


--- Top 5 Spas by Rating (min 5 reviews) ---
-------------------------------------------